In [2]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [2]:
def preprocess(df):
    le_cols = [
        '_1순위업종', '_2순위업종', '_3순위업종',
        '_1순위쇼핑업종', '_2순위쇼핑업종', '_3순위쇼핑업종',
        '_1순위교통업종', '_2순위교통업종', '_3순위교통업종',
        '_1순위여유업종', '_2순위여유업종', '_3순위여유업종',
        '_1순위납부업종', '_2순위납부업종', '_3순위납부업종',
        '최종카드론_신청경로코드', '이용금액대'
    ]

    for col in le_cols:
        df[col] = df[col].fillna(-1)
        codes, uniques = pd.factorize(df[col], sort=True)
        df[col] = codes

    return df
df1 = preprocess(pd.read_parquet('train/3.승인매출정보/201807_train_승인매출정보.parquet'))
df2 = preprocess(pd.read_parquet('train/3.승인매출정보/201808_train_승인매출정보.parquet'))
df3 = preprocess(pd.read_parquet('train/3.승인매출정보/201809_train_승인매출정보.parquet'))
df4 = preprocess(pd.read_parquet('train/3.승인매출정보/201810_train_승인매출정보.parquet'))
df5 = preprocess(pd.read_parquet('train/3.승인매출정보/201811_train_승인매출정보.parquet'))
df6 = preprocess(pd.read_parquet('train/3.승인매출정보/201812_train_승인매출정보.parquet'))

In [3]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

C:\Users\AHN\AppData\Local\Temp\ipykernel_10512\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_10512\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_10512\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

,ID,이용건수_페이_오프라인_B0M,쇼핑_기타_이용금액,연속무실적개월수_기본_24M_카드,할부건수_부분_14M_R12M,_2순위업종_이용금액,이용건수_할부_유이자_R3M,할부건수_부분_6M_R12M,증감_RP유형건수_전월,최종카드론_대출일자,...,이용금액_할부_무이자_R12M,_3순위업종,이용금액_오프라인_R3M,이용개월수_신용_R6M,납부_기타이용금액,_3순위여유업종,가맹점매출금액_B2M,RP후경과월_가스,이용금액_체크_R6M,RP건수_가스_B0M
0,TRAIN_000000,0.0,7.56250,0.00000,0.0,1399.34375,0.0000,0.0,0.00000,NaN,...,5444.15625,7.75,11301.65625,6.000,44.875,0.0,0.000,6.0,1125.34375,0.0
1,TRAIN_000001,0.0,327.03125,0.00000,0.0,1758.65625,0.0000,0.0,0.00000,20170327.0,...,2072.40625,2.25,12309.84375,6.000,0.000,0.0,0.000,6.0,0.00000,0.0
2,TRAIN_000002,0.0,0.00000,0.00000,0.0,1992.37500,0.0000,0.0,0.00000,20151119.0,...,10751.75000,2.00,21043.03125,6.000,0.000,0.0,5175.625,6.0,0.00000,0.0
3,TRAIN_000003,0.0,0.00000,0.00000,0.0,2020.12500,2.9375,0.0,0.03125,NaN,...,11151.59375,4.00,12449.06250,6.000,0.000,0.0,0.000,6.0,0.00000,0.0
4,TRAIN_000004,0.0,0.00000,3.31250,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00000,0.00,0.00000,0.625,0.000,0.0,0.000,6.0,11613.50000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.00000,5.43750,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00000,0.00,0.00000,0.000,0.000,0.0,0.000,6.0,13973.03125,0.0
399996,TRAIN_399996,0.0,347.78125,0.00000,0.0,2939.46875,0.0000,0.0,0.00000,20170112.0,...,1946.59375,4.00,28947.53125,6.000,171.500,0.0,0.000,6.0,0.00000,0.0
399997,TRAIN_399997,0.0,315.12500,0.00000,0.0,2281.78125,0.0000,0.0,0.00000,NaN,...,28323.00000,3.00,13575.93750,6.000,0.000,0.0,0.000,6.0,0.00000,0.0
399998,TRAIN_399998,0.0,0.00000,12.53125,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00000,0.00,0.00000,0.000,0.000,0.0,0.000,6.0,0.00000,0.0


In [4]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,이용건수_페이_오프라인_B0M,쇼핑_기타_이용금액,연속무실적개월수_기본_24M_카드,할부건수_부분_14M_R12M,_2순위업종_이용금액,이용건수_할부_유이자_R3M,할부건수_부분_6M_R12M,증감_RP유형건수_전월,최종카드론_대출일자,...,_3순위업종,이용금액_오프라인_R3M,이용개월수_신용_R6M,납부_기타이용금액,_3순위여유업종,가맹점매출금액_B2M,RP후경과월_가스,이용금액_체크_R6M,RP건수_가스_B0M,Segment
0,TRAIN_000000,0.0,7.56250,0.00000,0.0,1399.34375,0.0000,0.0,0.00000,NaN,...,7.75,11301.65625,6.000,44.875,0.0,0.000,6.0,1125.34375,0.0,D
1,TRAIN_000001,0.0,327.03125,0.00000,0.0,1758.65625,0.0000,0.0,0.00000,20170327.0,...,2.25,12309.84375,6.000,0.000,0.0,0.000,6.0,0.00000,0.0,E
2,TRAIN_000002,0.0,0.00000,0.00000,0.0,1992.37500,0.0000,0.0,0.00000,20151119.0,...,2.00,21043.03125,6.000,0.000,0.0,5175.625,6.0,0.00000,0.0,C
3,TRAIN_000003,0.0,0.00000,0.00000,0.0,2020.12500,2.9375,0.0,0.03125,NaN,...,4.00,12449.06250,6.000,0.000,0.0,0.000,6.0,0.00000,0.0,D
4,TRAIN_000004,0.0,0.00000,3.31250,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00,0.00000,0.625,0.000,0.0,0.000,6.0,11613.50000,0.0,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.00000,5.43750,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00,0.00000,0.000,0.000,0.0,0.000,6.0,13973.03125,0.0,E
399996,TRAIN_399996,0.0,347.78125,0.00000,0.0,2939.46875,0.0000,0.0,0.00000,20170112.0,...,4.00,28947.53125,6.000,171.500,0.0,0.000,6.0,0.00000,0.0,D
399997,TRAIN_399997,0.0,315.12500,0.00000,0.0,2281.78125,0.0000,0.0,0.00000,NaN,...,3.00,13575.93750,6.000,0.000,0.0,0.000,6.0,0.00000,0.0,C
399998,TRAIN_399998,0.0,0.00000,12.53125,0.0,0.00000,0.0000,0.0,0.00000,NaN,...,0.00,0.00000,0.000,0.000,0.0,0.000,6.0,0.00000,0.0,E


In [5]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
['최종카드론_대출일자', '최종카드론_금융상환방식코드']


In [6]:
ex1 = merged_df

In [8]:
cols_to_drop = ['최종카드론_대출일자', '최종카드론_금융상환방식코드']
ex1.drop(columns=cols_to_drop, inplace=True)

In [9]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['이용건수_페이_오프라인_B0M', '할부건수_부분_14M_R12M', '승인거절건수_입력오류_B0M', '이용건수_CA_R6M', '할부금액_무이자_14M_R12M', '이용건수_할부_유이자_R3M', '할부건수_부분_6M_R12M', '증감_RP유형건수_전월', '_2순위납부업종_이용금액', 'RP후경과월_건강', '할부건수_유이자_3M_R12M', '이용금액_당사페이_R6M', '증감_RP건수_학습비_전월', 'RP후경과월_렌탈', '신청건수_ATM_CL_B0', '여유_운동이용금액', '이용건수_C페이_B0M', '이용횟수_선결제_R3M', '증감_RP건수_보험_전월', '이용건수_C페이_R3M', '증감_RP건수_렌탈_전월', '이용건수_D페이_R3M', '교통_철도버스이용금액', '이용건수_부분무이자_R12M', '이용금액_카드론_B0M', '이용금액_부분무이자_R12M', '카드론이용월수_누적', '이용개월수_카드론_R3M', '이용건수_체크_R6M', '이용건수_부분무이자_R3M', '이용금액_B페이_B0M', '이용개월수_CA_R6M', '_3순위교통업종', '이용금액_할부_유이자_R3M', '할부건수_무이자_14M_R12M', 'RP후경과월_보험', '이용금액_카드론_R6M', '이용횟수_연체_R6M', '할부건수_유이자_12M_R12M', 'RP건수_건강_B0M', '이용건수_A페이_R3M', '_2순위여유업종', '이용후경과월_카드론', 'RP건수_학습비_B0M', '교통_통행료이용금액', '할부건수_부분_12M_R12M', '금액_할부전환_R3M', '여유_여행이용금액', '이용건수_CA_R12M', '이용금액_할부_유이자_B0M', '이용개월수_CA_R12M', '여유_기타이용금액', '할부금액_부분_12M_R12M', '이용개월수_부분무이자_R6M', '이용건수_C페이_R6M', '할부건수_유이자_14M_R12M', '최대이용금액_카드론_R12M', '이용금액_연체_B0M', '

In [10]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 825


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,이용건수_신용_R12M,이용건수_신판_R12M,0.999879,0.478777,0.474869
1,이용건수_신용_R6M,이용건수_신판_R6M,0.999841,0.438819,0.434845
2,이용건수_신판_R3M,이용건수_신용_R3M,0.999814,0.437778,0.441920
3,이용건수_신용_B0M,이용건수_신판_B0M,0.999804,0.443318,0.439110
4,이용건수_신판_R3M,이용건수_일시불_R3M,0.999781,0.437778,0.434641
...,...,...,...,...,...
820,이용건수_신판_B0M,이용금액_오프라인_B0M,0.800333,0.439110,0.588514
821,_3순위업종_이용금액,이용건수_일시불_R3M,0.800322,0.522730,0.434641
822,이용금액_온라인_R3M,이용건수_페이_온라인_B0M,0.800292,0.396921,0.337952
823,이용건수_신용_R6M,_3순위업종_이용금액,0.800219,0.438819,0.522730


In [11]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 126
제거할 피처 목록:
['연속무실적개월수_기본_24M_카드', '이용건수_간편결제_R3M', '_2순위업종_이용금액', '이용개월수_할부_유이자_R12M', '교통_주유이용금액', '이용건수_온라인_B0M', '쇼핑_마트_이용금액', '이용금액_온라인_R3M', '최종이용일자_신판', '납부_통신비이용금액', '이용금액대', '_1순위여유업종', '이용건수_오프라인_R3M', '이용건수_신용_B0M', '이용건수_신용_R6M', '이용금액_일시불_R3M', '할부금액_3M_R12M', '이용건수_오프라인_B0M', '정상청구원금_B0M', '이용건수_간편결제_R6M', '이용금액_부분무이자_R6M', '이용개월수_일시불_R6M', '이용개월수_오프라인_R6M', '이용건수_신판_B0M', '이용개월수_신판_R3M', '이용건수_할부_R12M', '이용개월수_신판_R12M', '이용금액_간편결제_R3M', '정상입금원금_B2M', 'RP건수_통신_B0M', '정상입금원금_B5M', '이용건수_할부_R6M', '이용금액_할부_무이자_R3M', 'RP유형건수_B0M', '이용개월수_할부_R12M', '이용금액_할부_무이자_R6M', '이용건수_일시불_R12M', '이용금액_간편결제_B0M', '이용개월수_전체_R6M', '연체입금원금_B2M', '이용건수_할부_R3M', '쇼핑_온라인_이용금액', '이용금액_일시불_R6M', '이용건수_페이_온라인_R6M', '이용개월수_신용_R3M', '이용금액_오프라인_B0M', '이용개월수_전체_R3M', '이용개월수_할부_R3M', '최대이용금액_할부_R12M', 'RP후경과월_통신', '할부금액_6M_R12M', '이용건수_페이_온라인_R3M', '이용개월수_일시불_R3M', '이용금액_할부_R3M', '이용후경과월_할부_무이자', '이용금액_오프라인_R6M', '이용개월수_할부_무이자_R12M', '최종이용일자_기본', '이용금액_페이_온라인_B0M', '쇼핑_슈퍼마켓_이용금액', '이용건수

In [12]:
cols_to_drop = ['연속무실적개월수_기본_24M_카드', '이용건수_간편결제_R3M', '_2순위업종_이용금액', '이용개월수_할부_유이자_R12M', '교통_주유이용금액', '이용건수_온라인_B0M', '쇼핑_마트_이용금액', '이용금액_온라인_R3M', '최종이용일자_신판', '납부_통신비이용금액', '이용금액대', '_1순위여유업종', '이용건수_오프라인_R3M', '이용건수_신용_B0M', '이용건수_신용_R6M', '이용금액_일시불_R3M', '할부금액_3M_R12M', '이용건수_오프라인_B0M', '정상청구원금_B0M', '이용건수_간편결제_R6M', '이용금액_부분무이자_R6M', '이용개월수_일시불_R6M', '이용개월수_오프라인_R6M', '이용건수_신판_B0M', '이용개월수_신판_R3M', '이용건수_할부_R12M', '이용개월수_신판_R12M', '이용금액_간편결제_R3M', '정상입금원금_B2M', 'RP건수_통신_B0M', '정상입금원금_B5M', '이용건수_할부_R6M', '이용금액_할부_무이자_R3M', 'RP유형건수_B0M', '이용개월수_할부_R12M', '이용금액_할부_무이자_R6M', '이용건수_일시불_R12M', '이용금액_간편결제_B0M', '이용개월수_전체_R6M', '연체입금원금_B2M', '이용건수_할부_R3M', '쇼핑_온라인_이용금액', '이용금액_일시불_R6M', '이용건수_페이_온라인_R6M', '이용개월수_신용_R3M', '이용금액_오프라인_B0M', '이용개월수_전체_R3M', '이용개월수_할부_R3M', '최대이용금액_할부_R12M', 'RP후경과월_통신', '할부금액_6M_R12M', '이용건수_페이_온라인_R3M', '이용개월수_일시불_R3M', '이용금액_할부_R3M', '이용후경과월_할부_무이자', '이용금액_오프라인_R6M', '이용개월수_할부_무이자_R12M', '최종이용일자_기본', '이용금액_페이_온라인_B0M', '쇼핑_슈퍼마켓_이용금액', '이용건수_간편결제_B0M', '_1순위업종_이용금액', '이용개월수_간편결제_R6M', '할부건수_무이자_3M_R12M', '_1순위쇼핑업종_이용금액', '이용건수_체크_R12M', 'RP건수_교통_B0M', '이용개월수_체크_R12M', '이용후경과월_신용', '이용건수_일시불_R6M', '이용건수_온라인_R3M', 'RP후경과월', '이용건수_할부_무이자_R6M', '연체입금원금_B5M', '이용건수_신판_R6M', '이용후경과월_신판', '_1순위쇼핑업종', '쇼핑_도소매_이용금액', '이용건수_일시불_B0M', '이용개월수_결제일_R6M', '최대이용금액_일시불_R12M', '이용개월수_할부_R6M', '이용개월수_온라인_R6M', '이용개월수_페이_온라인_R6M', '이용개월수_신용_R12M', '_3순위쇼핑업종_이용금액', '쇼핑_편의점_이용금액', '이용건수_신판_R12M', '이용개월수_신판_R6M', '이용건수_신용_R12M', '교통_버스지하철이용금액', '이용건수_페이_온라인_B0M', '이용후경과월_할부_유이자', '이용개월수_할부_무이자_R3M', '이용금액_온라인_B0M', '이용금액_페이_온라인_R6M', '이용건수_일시불_R3M', '이용개월수_할부_무이자_R6M', '정상청구원금_B2M', '최대이용금액_체크_R12M', '이용금액_간편결제_R6M', '이용건수_할부_무이자_R3M', '이용금액_할부_R6M', '_3순위업종_이용금액', '이용건수_온라인_R6M', '_2순위쇼핑업종_이용금액', '이용건수_신판_R3M', '이용가맹점수', '이용후경과월_할부', '할부건수_3M_R12M', '할부금액_무이자_3M_R12M', '이용개월수_일시불_R12M', '이용건수_신용_R3M', '이용금액_페이_온라인_R3M', '이용금액_일시불_B0M', '이용금액_일시불_R12M', '할부금액_무이자_6M_R12M', '이용건수_오프라인_R6M', '이용개월수_결제일_R3M', '정상입금원금_B0M', '이용금액_할부_무이자_R12M', '이용금액_오프라인_R3M', '이용개월수_신용_R6M', '이용금액_온라인_R6M', '이용건수_할부_무이자_R12M', '이용금액_할부_유이자_R12M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [13]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 '쇼핑_기타_이용금액',
 '정상청구원금_B5M',
 '최대이용금액_할부_무이자_R12M',
 '이용후경과월_일시불',
 '증감_RP건수_전월',
 '최종이용일자_CA',
 '교통_택시이용금액',
 '_2순위업종',
 '연속유실적개월수_기본_24M_카드',
 '_1순위교통업종',
 '_1순위납부업종',
 '최종이용일자_일시불',
 '최대이용금액_할부_유이자_R12M',
 '이용금액_부분무이자_R3M',
 '최종이용일자_할부',
 '_1순위교통업종_이용금액',
 'RP금액_B0M',
 '연체입금원금_B0M',
 '이용금액_할부_R12M',
 '_1순위업종',
 'RP건수_B0M',
 '_1순위납부업종_이용금액',
 '이용금액_체크_R12M',
 '교통_정비이용금액',
 '_1순위여유업종_이용금액',
 '_2순위교통업종',
 'RP후경과월_교통',
 '최종이용일자_체크',
 '_3순위쇼핑업종',
 '_2순위교통업종_이용금액',
 '_2순위쇼핑업종',
 '_3순위업종',
 '납부_기타이용금액',
 'Segment']

In [14]:
ex1.to_parquet('승인_전처리_Segment.parquet', index=False)

In [3]:
def preprocess(df):
    le_cols = [
        '_1순위업종', '_2순위업종', '_3순위업종',
        '_1순위쇼핑업종', '_2순위쇼핑업종', '_3순위쇼핑업종',
        '_1순위교통업종', '_2순위교통업종', '_3순위교통업종',
        '_1순위여유업종', '_2순위여유업종', '_3순위여유업종',
        '_1순위납부업종', '_2순위납부업종', '_3순위납부업종',
        '최종카드론_신청경로코드', '이용금액대'
    ]

    for col in le_cols:
        df[col] = df[col].fillna(-1)
        codes, uniques = pd.factorize(df[col], sort=True)
        df[col] = codes

    return df


ddf1 = preprocess(pd.read_parquet('train/3.승인매출정보/201807_train_승인매출정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/3.승인매출정보/201808_train_승인매출정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/3.승인매출정보/201809_train_승인매출정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/3.승인매출정보/201810_train_승인매출정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/3.승인매출정보/201811_train_승인매출정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/3.승인매출정보/201812_train_승인매출정보.parquet'))

In [4]:
dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

C:\Users\AHN\AppData\Local\Temp\ipykernel_23420\2019321182.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[base_col] = merged_df[cols].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_23420\2019321182.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[base_col] = merged_df[cols].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_23420\2019321182.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

             ID     최종이용일자_기본     최종이용일자_신판     최종이용일자_CA  최종이용일자_카드론  \
0  TRAIN_000000  2.018101e+07  2.018101e+07  2.018101e+07     10101.0   
1  TRAIN_000001  2.018101e+07  2.018101e+07  2.017073e+07  20170327.0   
2  TRAIN_000002  2.018101e+07  2.018101e+07  2.018100e+07  20151119.0   
3  TRAIN_000003  2.018101e+07  2.018100e+07  2.018101e+07     10101.0   
4  TRAIN_000004  2.018072e+07  2.018072e+07  1.010100e+04     10101.0   

      최종이용일자_체크    최종이용일자_일시불     최종이용일자_할부  이용건수_신용_B0M  이용건수_신판_B0M  ...  \
0  2.018020e+07  2.018101e+07  2.018071e+07     7.153846     5.153846  ...   
1  1.010100e+04  2.018101e+07  2.017123e+07    10.884615    10.884615  ...   
2  2.014123e+07  2.018100e+07  2.018081e+07    18.538462    16.538462  ...   
3  2.014111e+07  2.018100e+07  2.018100e+07     9.461538     7.461538  ...   
4  2.018101e+07  2.018072e+07  1.010100e+04    -0.269231    -0.269231  ...   

   승인거절건수_한도초과_B0M  승인거절건수_BL_B0M  승인거절건수_입력오류_B0M  승인거절건수_기타_B0M  승인거절건수_R3M  \
0          

In [5]:
cols = ['ID',
 '쇼핑_기타_이용금액',
 '정상청구원금_B5M',
 '최대이용금액_할부_무이자_R12M',
 '이용후경과월_일시불',
 '증감_RP건수_전월',
 '최종이용일자_CA',
 '교통_택시이용금액',
 '_2순위업종',
 '연속유실적개월수_기본_24M_카드',
 '_1순위교통업종',
 '_1순위납부업종',
 '최종이용일자_일시불',
 '최대이용금액_할부_유이자_R12M',
 '이용금액_부분무이자_R3M',
 '최종이용일자_할부',
 '_1순위교통업종_이용금액',
 'RP금액_B0M',
 '연체입금원금_B0M',
 '이용금액_할부_R12M',
 '_1순위업종',
 'RP건수_B0M',
 '_1순위납부업종_이용금액',
 '이용금액_체크_R12M',
 '교통_정비이용금액',
 '_1순위여유업종_이용금액',
 '_2순위교통업종',
 'RP후경과월_교통',
 '최종이용일자_체크',
 '_3순위쇼핑업종',
 '_2순위교통업종_이용금액',
 '_2순위쇼핑업종',
 '_3순위업종',
 '납부_기타이용금액']

result = result[cols]
result

,ID,쇼핑_기타_이용금액,정상청구원금_B5M,최대이용금액_할부_무이자_R12M,이용후경과월_일시불,증감_RP건수_전월,최종이용일자_CA,교통_택시이용금액,_2순위업종,연속유실적개월수_기본_24M_카드,...,교통_정비이용금액,_1순위여유업종_이용금액,_2순위교통업종,RP후경과월_교통,최종이용일자_체크,_3순위쇼핑업종,_2순위교통업종_이용금액,_2순위쇼핑업종,_3순위업종,납부_기타이용금액
0,TRAIN_000000,9.307692,15945.500000,0.000000,0.000000,0.192308,2.018101e+07,208.269231,5.000000,16.076923,...,0.000000,0.000000,0.038462,4.692308,2.018020e+07,0.000000,4.038462,0.000000,5.923077,44.192308
1,TRAIN_000001,316.500000,4419.307692,3161.576923,0.000000,0.000000,2.017073e+07,0.000000,3.961538,13.923077,...,0.000000,0.000000,2.461538,6.000000,1.010100e+04,6.269231,266.269231,4.615385,2.961538,0.000000
2,TRAIN_000002,0.000000,23260.884615,13696.461538,0.000000,0.000000,2.018100e+07,0.000000,4.000000,9.153846,...,0.000000,0.000000,0.000000,6.000000,2.014123e+07,1.615385,0.000000,1.384615,2.000000,0.000000
3,TRAIN_000003,0.000000,18750.538462,7451.884615,0.000000,0.038462,2.018101e+07,206.461538,4.153846,12.884615,...,0.000000,0.000000,1.923077,3.884615,2.014111e+07,0.961538,83.461538,4.423077,4.000000,0.000000
4,TRAIN_000004,0.000000,66.538462,0.000000,2.730769,0.000000,1.010100e+04,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,6.000000,2.018101e+07,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.000000,120.346154,0.000000,0.461538,0.000000,1.010100e+04,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,6.000000,2.018100e+07,0.000000,0.000000,0.000000,0.000000,0.000000
399996,TRAIN_399996,349.807692,23919.923077,3811.576923,0.000000,0.000000,1.010100e+04,208.307692,7.307692,17.884615,...,0.000000,0.000000,4.769231,0.000000,1.010100e+04,4.423077,262.269231,4.153846,4.000000,194.384615
399997,TRAIN_399997,313.769231,8974.576923,16416.346154,0.000000,0.000000,1.010100e+04,0.000000,2.384615,24.000000,...,392.846154,949.346154,2.000000,6.000000,2.013112e+07,4.000000,392.846154,5.000000,3.615385,0.000000
399998,TRAIN_399998,0.000000,296.807692,0.000000,12.000000,0.000000,1.010100e+04,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,6.000000,1.010100e+04,0.000000,0.000000,0.000000,0.000000,0.000000


In [6]:
result.to_parquet('승인_전처리_test.parquet', index=False)

In [7]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
